In [1]:
# ⚙️ Global Config & Services (using centralized modules)

import json
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings,
    validate_api_keys,
    print_config_summary
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


📂 Working directory: d:\Madhura\ai_eng\mini_project_1\rag
✅ Config loaded:
  LLM: groq / llama-3.1-8b-instant
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


d:\Madhura\ai_eng\mini_project_1\rag\src\services\llm_services.py:375: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
d:\Madhura\ai_eng\mini_project_1\rag\src\services\llm_services.py:375: UserWarning: ⚠️  GOOGLE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
d:\Madhura\ai_eng\mini_project_1\rag\src\services\llm_services.py:375: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


In [2]:
# Initialize LLM, Embeddings, and Reranker
from sentence_transformers import CrossEncoder

llm = get_llm(config)
embeddings = get_text_embeddings(config)

# CrossEncoder: A reranker model that scores query-document pairs
# Unlike bi-encoders (embeddings), cross-encoders see query AND document together
# This gives higher accuracy but is slower (can't pre-compute embeddings)
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"  # Model trained on MS MARCO dataset
                                             # Other options: "cross-encoder/ms-marco-TinyBERT-L-2-v2" (faster)
                                             #                "cross-encoder/ms-marco-MiniLM-L-12-v2" (more accurate)
)

print(f"✅ LLM: {config['llm_provider']} / {config.get('openrouter_model', config.get('llm_model'))}")
print(f"✅ Embeddings: {config['text_emb_model']}")
print(f"✅ Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ LLM API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")


d:\Madhura\ai_eng\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Madhura\ai_eng\mini_project_1\rag\src\services\llm_services.py:129: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 852.13it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-


✅ LLM: groq / gpt-4o-mini
✅ Embeddings: sentence-transformers/all-MiniLM-L6-v2
✅ Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

🔍 Testing LLM API connection...
✅ LLM API verified: API working!


---

### Step 1: Load or Create Data


In [4]:
from langchain_community.document_loaders import (
                                                PyPDFLoader, 
                                                DirectoryLoader, 
                                                TextLoader
                                                )
from langchain_core.documents import Document

pdf_dir = Path(config["data_root"]) / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)

# Try loading PDFs
pdf_files = list(pdf_dir.glob("*.pdf"))

if len(pdf_files) == 0:
    print("⚠️  No PDFs found. Creating sample text document...")
    
    sample_content = """# Uber Inc. Annual Business Overview

Understanding Uber's business model and financial performance is essential for evaluating its long-term growth strategy. Uber operates a global technology platform that connects consumers with drivers, merchants, and logistics providers.

## Core Business Segments

### Mobility
Mobility represents Uber's ride-hailing services, enabling consumers to request transportation via the Uber app. The segment operates in thousands of cities worldwide. Revenue is generated primarily through service fees charged to drivers. Growth in this segment is driven by increased trip volume, geographic expansion, and pricing optimization.

### Delivery
Delivery includes Uber Eats and grocery services, connecting consumers with restaurants and retailers. The platform benefits from cross-platform engagement, where users who utilize both Mobility and Delivery generate higher monthly transaction frequency. Revenue growth is supported by advertising services and merchant partnerships.

## Financial Performance Drivers

### Network Effects
Uber's platform becomes more valuable as more drivers, consumers, and merchants join the ecosystem. Increased liquidity improves matching efficiency and reduces wait times, enhancing overall customer experience.

### Pricing and Marketplace Technology
Uber utilizes demand prediction, dynamic pricing, and routing algorithms to optimize marketplace efficiency. These proprietary technologies help balance supply and demand across different regions and time periods.

## Advertising and Ancillary Revenue

### Advertising
Uber leverages its platform data and scale to offer marketplace-centric advertising solutions. Brands can engage consumers throughout their journey within the app. Advertising provides a high-margin revenue stream that complements core Mobility and Delivery operations.

### Membership Programs
Uber One is a cross-platform membership offering discounts, priority service, and exclusive benefits across ride-hailing and delivery services. Membership programs aim to increase retention, engagement, and recurring revenue.

## Competitive Environment

Uber operates in highly competitive and fragmented markets globally. It faces competition from local ride-hailing providers, food delivery platforms, and logistics companies. Competitive pressures may impact pricing power and margins.

## Risk Factors

Operational risks include regulatory changes, driver classification policies, cybersecurity threats, and macroeconomic conditions affecting consumer demand. Strategic investments in technology and expansion are intended to mitigate these risks and strengthen long-term growth.
"""
    
    sample_file = pdf_dir / "financial_sample.txt"
    sample_file.write_text(sample_content)
    
    # Load text file as document
    documents = [Document(page_content=sample_content, metadata={"source": "financial_sample.txt", "page": 0})]
    
else:
    # Load PDFs
    documents = []
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        documents.extend(docs)

print(f"✅ Loaded {len(documents)} document pages")
print(f"  Total characters: {sum(len(d.page_content) for d in documents):,}")

✅ Loaded 142 document pages
  Total characters: 639,466


---

### Step 2: Dense Retrieval (ChromaDB)

Build a vector store using dense embeddings.


In [15]:
from langchain_chroma import Chroma

# Setup persistence directory for ChromaDB
chroma_root = Path(config["artifacts_root"]) / "chroma"
chroma_root.mkdir(parents=True, exist_ok=True)

print("🔵 Building dense vector store...")

# Chroma.from_documents: Creates a vector store from LangChain Documents
dense_vectorstore = Chroma.from_documents(
    documents=documents,        # documents: List of Document objects to index
    embedding=embeddings,       # embedding: Embedding model to convert text → vectors
    collection_name="advanced_dense",  # collection_name: Name of the collection in ChromaDB
    persist_directory=str(chroma_root / "advanced_dense"),  # persist_directory: Where to save the index
)

print(f"✅ Dense index built: {len(documents)} docs")

# Test dense retrieval
query = "What services are included under Uber's platform offerings?"

# similarity_search: Find documents with vectors closest to query vector
dense_results = dense_vectorstore.similarity_search(
    query,  # query: Search query (will be embedded automatically)
    k=8     # k: Number of top results to return
)

# Add doc_id to dense results (find index in original documents list)
for doc in dense_results:
    doc_id = next((i for i, d in enumerate(documents) if d.page_content == doc.page_content), None)
    if doc_id is not None:
        doc.metadata["doc_id"] = doc_id

print(f"\n🔍 Dense search: '{query}'")
for i, doc in enumerate(dense_results, 1):
    print(f"  [{i}] {doc.page_content[:100]}...")

🔵 Building dense vector store...
✅ Dense index built: 142 docs

🔍 Dense search: 'What services are included under Uber's platform offerings?'
  [1] 82 
UBER TECHNOLOGIES, INC.  
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS 
Note 1 – Description of Bu...
  [2] 82 
UBER TECHNOLOGIES, INC.  
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS 
Note 1 – Description of Bu...
  [3] 82 
UBER TECHNOLOGIES, INC.  
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS 
Note 1 – Description of Bu...
  [4] 4 
PART I 
ITEM 1. BUSINESS 
Overview 
Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” ...
  [5] 4 
PART I 
ITEM 1. BUSINESS 
Overview 
Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” ...
  [6] 4 
PART I 
ITEM 1. BUSINESS 
Overview 
Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” ...
  [7] 75 
UBER TECHNOLOGIES, INC. 
CONSOLIDATED STATEMENTS OF OPERATIONS 
(In millions, except share amoun...
  [8] 75 
UBER TECHNOLOGIES, INC. 
CONSOLIDATED STATEMENTS OF OPERATIONS 
(In millions, 

---

### Step 3: Sparse Retrieval (BM25)

Use BM25 for keyword-based retrieval.


In [7]:
from rank_bm25 import BM25Okapi
import numpy as np

print("🟠 Building BM25 index...")

# BM25 (Best Match 25): Classic sparse retrieval algorithm
# Unlike dense retrieval, BM25 uses exact keyword matching with TF-IDF-like scoring

# Step 1: Tokenize corpus (lowercase + split by whitespace)
tokenized_corpus = [doc.page_content.lower().split() for doc in documents]

# BM25Okapi: BM25 variant with Okapi weighting
# Other variants: BM25L, BM25Plus (handle long documents better)
bm25 = BM25Okapi(tokenized_corpus)

print(f"✅ BM25 index built")


def bm25_search(query: str, top_k: int = 3):
    """
    Search using BM25 (sparse retrieval algorithm).
    
    Args:
        query: Search query string
        top_k: Number of top results to return
        
    Returns:
        List of dictionaries with doc, score, and doc_id
    """
    # Tokenize query the same way as corpus
    tokenized_query = query.lower().split()
    
    # Get BM25 scores for all documents
    scores = bm25.get_scores(tokenized_query)
    
    # Find top-k indices (argsort ascending, then reverse for descending)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # Build results list
    results = []
    for idx in top_indices:
        results.append({
            "doc": documents[idx],
            "score": float(scores[idx]),
            "doc_id": idx
        })
    
    return results

# Test BM25
bm25_results = bm25_search(query, top_k=8)

print(f"\n🔍 BM25 search: '{query}'")
for i, res in enumerate(bm25_results, 1):
    print(f"  [{i}] (score: {res['score']:.2f}) {res['doc'].page_content[:100]}...")


🟠 Building BM25 index...
✅ BM25 index built

🔍 BM25 search: 'What services are included under Uber's platform offerings?'
  [1] (score: 7.60) Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The ...
  [2] (score: 6.82) 94 
15, 2026, and interim periods within fiscal years beginning afte r December 15, 2027. Early adop...
  [3] (score: 6.72) 31 
mergers, and sales of assets, and restrictions on the payment of  dividends or distributions. An...
  [4] (score: 6.23) 84 
Property and Equipment, Net  
Property and equipment are stated at cost, net of accumulated depr...
  [5] (score: 5.94) 64 
Debt and Revolving Credit Arrangements in the notes to the consolid ated financial statements in...
  [6] (score: 5.66) 19 
laws, and regulations; 
• laws and regulations more restrictive than those in the United States,...
  [7] (score: 5.58) 88 
In markets where we agree to provide Mobility or Delivery servic es to end-users for a fee, we a...
  [8] (sco

---

### Step 4: Hybrid Fusion (Dense + BM25)

Combine dense and sparse retrieval using Reciprocal Rank Fusion (RRF).


In [16]:
from typing import List

def rrf_fusion(dense_docs: List, bm25_results: List, k: int = 60) -> List:
    """
    Reciprocal Rank Fusion (RRF) - combines dense and sparse retrieval.
    
    RRF Formula: score(d) = Σ 1/(k + rank(d)) for each ranking
    
    - k=60 is the default constant (from original paper)
    - Higher k → more weight to lower-ranked documents
    - Lower k → more weight to top-ranked documents
    
    Args:
        dense_docs: Results from dense (vector) retrieval
        bm25_results: Results from BM25 (sparse) retrieval
        k: RRF constant (default 60, typical range 1-100)
        
    Returns:
        Fused results sorted by RRF score (higher = more relevant)
    """
    rrf_scores = {}
    
    # Add scores from dense retrieval
    for rank, doc in enumerate(dense_docs, 1):  # rank starts at 1
        doc_id = doc.metadata["doc_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    
    # Add scores from BM25 retrieval
    for rank, doc in enumerate(bm25_results, 1):
        doc_id = doc["doc_id"]
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    
    # Sort by combined RRF score (descending)
    sorted_ids = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Build final results list
    fused_docs = []
    for doc_id, score in sorted_ids:
        fused_docs.append({
            "doc": documents[doc_id],
            "score": score,
            "doc_id": doc_id
        })
    
    return fused_docs
# Test hybrid fusion
fused_results = rrf_fusion(dense_results, bm25_results)

print(f"🔀 Hybrid (RRF) search: '{query}'")
for i, res in enumerate(fused_results, 1):
    print(f"  [{i}] (RRF: {res['score']:.3f}) {res['doc'].page_content[:100]}...")


🔀 Hybrid (RRF) search: 'What services are included under Uber's platform offerings?'
  [1] (RRF: 0.048) 82 
UBER TECHNOLOGIES, INC.  
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS 
Note 1 – Description of Bu...
  [2] (RRF: 0.046) 4 
PART I 
ITEM 1. BUSINESS 
Overview 
Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” ...
  [3] (RRF: 0.030) 75 
UBER TECHNOLOGIES, INC. 
CONSOLIDATED STATEMENTS OF OPERATIONS 
(In millions, except share amoun...
  [4] (RRF: 0.016) Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The ...
  [5] (RRF: 0.016) 94 
15, 2026, and interim periods within fiscal years beginning afte r December 15, 2027. Early adop...
  [6] (RRF: 0.016) 31 
mergers, and sales of assets, and restrictions on the payment of  dividends or distributions. An...
  [7] (RRF: 0.016) 84 
Property and Equipment, Net  
Property and equipment are stated at cost, net of accumulated depr...
  [8] (RRF: 0.015) 64 
Debt and Revolving Credit Arran

---

### Step 5: Reranking with Cross-Encoder

Refine results using a cross-encoder for more accurate relevance scoring.


In [17]:
def rerank(query: str, results: List, top_k: int = 3):
    """
    Rerank results using a cross-encoder for more accurate relevance scoring.
    
    Cross-encoder sees [query, document] together, enabling deeper understanding
    of relevance than separate embeddings.
    
    Args:
        query: Search query string
        results: List of initial results to rerank (from fusion)
        top_k: Number of top results to return after reranking
        
    Returns:
        Reranked results with rerank_score added (higher = more relevant)
    """
    # Create query-document pairs for cross-encoder
    # Format: [[query, doc1], [query, doc2], ...]
    pairs = [[query, res['doc'].page_content] for res in results]
    
    # Cross-encoder predicts relevance score for each pair
    # Returns array of scores (can be negative, higher = more relevant)
    scores = reranker.predict(pairs)
    
    # Add rerank scores to results
    for i, res in enumerate(results):
        res['rerank_score'] = float(scores[i])
    
    # Sort by rerank score (descending) and take top-k
    reranked = sorted(results, key=lambda x: x['rerank_score'], reverse=True)[:top_k]
    return reranked

# Test reranking
reranked_results = rerank(query, fused_results[:6], top_k=3)

print(f"🏆 Reranked results: '{query}'")
for i, res in enumerate(reranked_results, 1):
    print(f"  [{i}] (rerank: {res['rerank_score']:.3f}) {res['doc'].page_content[:100]}...")


🏆 Reranked results: 'What services are included under Uber's platform offerings?'
  [1] (rerank: 4.220) 4 
PART I 
ITEM 1. BUSINESS 
Overview 
Uber Technologies, Inc. (“Uber,” the “Company,” “we,” “our,” ...
  [2] (rerank: 2.624) 82 
UBER TECHNOLOGIES, INC.  
NOTES TO CONSOLIDATED FINANCIAL STATEMENTS 
Note 1 – Description of Bu...
  [3] (rerank: -2.124) 94 
15, 2026, and interim periods within fiscal years beginning afte r December 15, 2027. Early adop...


In [20]:
def hybrid_rag_pipeline(
    query: str, 
    dense_top_n: int = 10,   # dense_top_n: How many docs to retrieve with dense search
    bm25_top_n: int = 10,    # bm25_top_n: How many docs to retrieve with BM25
    rerank_top_n: int = 6,   # rerank_top_n: How many fused results to rerank
    final_top_k: int = 3     # final_top_k: Final number of docs for LLM context
):
    """
    Complete hybrid RAG pipeline combining all advanced techniques.
    
    Pipeline: Dense → BM25 → Fusion (RRF) → Rerank → LLM Generation
    
    Args:
        query: User question
        dense_top_n: Number of results from dense retrieval
        bm25_top_n: Number of results from BM25
        rerank_top_n: Number of fused results to rerank (reduces cross-encoder calls)
        final_top_k: Final number of chunks to use for generation
        
    Returns:
        Dictionary with query, answer, retrieved_docs, and pipeline stats
    """
    # Stage 1: Dense retrieval (semantic similarity)
    dense_results = dense_vectorstore.similarity_search(query, k=dense_top_n)
    
    # Add doc_id to dense results
    for doc in dense_results:
        doc_id = next((i for i, d in enumerate(documents) if d.page_content == doc.page_content), None)
        if doc_id is not None:
            doc.metadata["doc_id"] = doc_id
    
    # Stage 2: Sparse retrieval (keyword matching)
    bm25_results = bm25_search(query, top_k=bm25_top_n)
    
    # Stage 3: Fusion (combine rankings with RRF)
    fused_results = rrf_fusion(dense_results, bm25_results)[:rerank_top_n]
    
    # Stage 4: Reranking (refine with cross-encoder)
    reranked_results = rerank(query, fused_results, top_k=final_top_k)
    
    # Stage 5: Build context from top results
    context = "\n\n".join([res["doc"].page_content for res in reranked_results])
    
    # Stage 6: Generate answer with LLM
    prompt = f"""Use the following context to answer the question. Be concise and accurate.

Context:
{context}

Question: {query}

Answer:"""
    
    response = llm.invoke(prompt)
    answer = response.content if hasattr(response, 'content') else str(response)
    
    return {
        "query": query,
        "answer": answer,
        "retrieved_docs": reranked_results,
        "num_dense": len(dense_results),
        "num_bm25": len(bm25_results),
        "num_fused": len(fused_results),
        "num_final": len(reranked_results)
    }


# Test the complete pipeline
print("🚀 Testing Complete Hybrid RAG Pipeline\n")
test_query = "What services are included under Uber's platform offerings?"
result = hybrid_rag_pipeline(test_query)

print(f"Query: {test_query}")
print(f"\nPipeline stats:")
print(f"  Dense retrieval: {result['num_dense']} docs")
print(f"  BM25 retrieval: {result['num_bm25']} docs")
print(f"  After fusion: {result['num_fused']} docs")
print(f"  After reranking: {result['num_final']} docs")
print(f"\nFinal Answer:\n{result['answer']}")

🚀 Testing Complete Hybrid RAG Pipeline

Query: What services are included under Uber's platform offerings?

Pipeline stats:
  Dense retrieval: 10 docs
  BM25 retrieval: 10 docs
  After fusion: 6 docs
  After reranking: 3 docs

Final Answer:
According to the provided context, Uber's platform offerings include the following services:

1. Mobility services:
   - Ridesharing
   - Carsharing
   - Micromobility
   - Rentals
   - Public transit
   - Taxis

2. Delivery services:
   - Meal preparation
   - Grocery delivery
   - Delivery of items from restaurants, grocers, and other retailers
   - Uber Direct (white-label Delivery-as-a-Service offering to retailers and restaurants)

3. Freight services:
   - Freight transportation services
   - Transportation management

4. Other services:
   - Financial partnerships products
   - Advertising services (sponsored listing fees paid by Merchants and brands)
   - Public transportation connections
   - Subscription memberships (Uber One, Uber Pass, R